In [1]:
# Ячейка 1 - Импорт
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import accuracy_score
import pickle

# Загружаем данные
X_train = np.load('../../data/X_train.npy')
X_test = np.load('../../data/X_test.npy')
y_train = np.load('../../data/y_train.npy')
y_test = np.load('../../data/y_test.npy')

with open('../../data/label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

NUM_CLASSES = len(le.classes_)
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_test_cat = to_categorical(y_test, NUM_CLASSES)

print(f"Данные загружены!")
print(f"Train: {X_train.shape}")
print(f"Классов: {NUM_CLASSES}")

Данные загружены!
Train: (638376, 10, 5)
Классов: 10


In [2]:
# Ячейка 2 - Улучшенная архитектура
inputs = Input(shape=(10, 5))

# LSTM слои
x = LSTM(128, return_sequences=True)(inputs)
x = Dropout(0.2)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.2)(x)

# Attention механизм
attention = MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
x = LayerNormalization()(attention + x)

# Финальные слои
x = GlobalAveragePooling1D()(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.2)(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)

model_v2 = Model(inputs, outputs)
model_v2.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_v2.summary()

2026-04-16 20:05:38.219331: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-04-16 20:05:38.219523: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-04-16 20:05:38.219530: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-04-16 20:05:38.219870: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-16 20:05:38.219953: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 10, 5)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 10, 128)   │     68,608 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 10, 128)   │          0 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 10, 64)    │     49,408 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 10, 64)    │          0 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 10, 64)    │     16,640 │ dropout_1[0][0],  │
│ (MultiHeadAttentio… │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 10, 64)    │          0 │ multi_head_atten… │
│                     │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 10, 64)    │        128 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      4,160 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 10)        │        650 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 139,594 (545.29 KB)

 Trainable params: 139,594 (545.29 KB)

 Non-trainable params: 0 (0.00 B)

In [3]:
# Ячейка 3 - Обучение улучшенной модели
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=3, verbose=1)
]

history = model_v2.fit(
    X_train, y_train_cat,
    epochs=30,
    batch_size=512,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/30


2026-04-16 20:06:04.342025: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1123/1123 ━━━━━━━━━━━━━━━━━━━━ 59s 46ms/step - accuracy: 0.5763 - loss: 1.3083 - val_accuracy: 0.6767 - val_loss: 0.9749 - learning_rate: 0.0010
Epoch 2/30
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 50s 44ms/step - accuracy: 0.6730 - loss: 0.9938 - val_accuracy: 0.6887 - val_loss: 0.9158 - learning_rate: 0.0010
Epoch 3/30
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 49s 44ms/step - accuracy: 0.6879 - loss: 0.9376 - val_accuracy: 0.7022 - val_loss: 0.8822 - learning_rate: 0.0010
Epoch 4/30
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 50s 44ms/step - accuracy: 0.6953 - loss: 0.9108 - val_accuracy: 0.7062 - val_loss: 0.8538 - learning_rate: 0.0010
Epoch 5/30
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 50s 45ms/step - accuracy: 0.6983 - loss: 0.8981 - val_accuracy: 0.7139 - val_loss: 0.8387 - learning_rate: 0.0010
Epoch 6/30
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 54s 48ms/step - accuracy: 0.6999 - loss: 0.8894 - val_accuracy: 0.7058 - val_loss: 0.8495 - learning_rate: 0.0010
Epoch 7/30
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 56s 50ms/step - accuracy: 0.7012 

In [4]:
# Ячейка 4 - Оценка улучшенной модели
y_pred_v2 = model_v2.predict(X_test[:10000], verbose=0).argmax(axis=1)
acc_v2 = accuracy_score(y_test[:10000], y_pred_v2)

print(f"Базовая модель:    70.0%")
print(f"Улучшенная модель: {acc_v2:.2%}")
print(f"Улучшение:         +{(acc_v2 - 0.70)*100:.1f}%")

# Сохраняем улучшенную модель
import os
os.makedirs('../../models', exist_ok=True)
model_v2.save('../../models/lstm_model_v2.keras')
print("\nМодель v2 сохранена!")

Базовая модель:    70.0%
Улучшенная модель: 71.84%
Улучшение:         +1.8%

Модель v2 сохранена!
